In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import matplotlib.dates as mdates
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import seaborn as sns

In [3]:
data = "/content/Data_Unicharm.xlsx - Data.csv"
df = pd.read_csv(data, thousands=',')


df = df.sort_values("Date").reset_index(drop=True)

print(f"Dataset loader: {len(df)} quaraters of data")
display(df)

Dataset loader: 34 quaraters of data


,Date,Revenue,Operating Income,Net Profit,COGS,Sales of daily consumer goods,Diapers Exp,Cosmetics,Cardboard,CPI_YoY,Exchange_Rate
0,2013/04/01-2013/06/30,143761,17652,10949,78440,8389206,3456,3116,6.5,0,104
1,2013/07/01-2013/09/30,142857,14393,6388,79654,8258224,3468,3075,7.5,-1,100
2,2013/10/01-2013/12/31,152360,17150,12013,83983,9705282,3479,3378,8.5,-1,98
3,2014/04/01-2014/06/30,195709,18517,9080,110729,8618481,3498,2848,9.0,-1,95
4,2014/07/01-2014/09/30,161117,18191,13612,89221,8761662,3516,3059,9.5,0,92
5,2015/01/01-2015/03/31,177558,17049,5778,99327,8756885,3502,2948,10.0,0,96
6,2015/04/01-2015/06/30,181251,17651,10113,100577,8934962,3502,3226,10.0,1,94
7,2015/07/01-2015/09/30,170262,15178,1685,94138,8914209,3502,4991,10.0,1,96
8,2016/01/01-2016/03/31,173065,15039,8517,94077,8690579,3422,4453,10.0,1,95
9,2016/04/01-2016/06/30,177414,20734,9038,93684,8576261,3422,4839,10.0,1,95


In [4]:
df["Date"] = df["Date"].str.split("-").str[1]
df["Date"] = pd.to_datetime(df["Date"])

In [5]:
df = df.sort_values("Date").reset_index(drop=True)

def create_features(data):
  df_feat = data.copy()

  # Feature 1: Seasonality (Quarter 1-4)
  # Divide Quarters
  df_feat["Quarter"] = df_feat["Date"].dt.quarter

  # Feature 2: Lags (Past Performance)
  # The best predictor of tomorrow is often todat (Lag 1) or this time last year
  for col in ["Revenue", "Operating Income", "Net Profit"]:
    df_feat[f"{col}_Lag1"] = df_feat[col].shift(1) # Previous Quarter
    df_feat[f"{col}_Lag4"] = df_feat[col].shift(4) # Same Quarter Last Year

  # B. MACRO LAGS (Your Request: "Known at time t")
    # We shift these so we predict T using data from T-1
  macros = ['Diapers Exp', 'Cosmetics', 'CPI_YoY', 'Exchange_Rate', "Cardboard", "Sales of daily consumer goods", "COGS"]
  for col in macros:
      df_feat[f'{col}_Lag1'] = df_feat[col].shift(1)

  # Drop the first 4 rows because they now have NaNs (no history for them)
  df_feat = df_feat.dropna().reset_index(drop=True)

  return df_feat

df_model = create_features(df)


In [6]:
features_rev = ["Quarter", "Diapers Exp_Lag1", "Cosmetics_Lag1", "Cardboard_Lag1", "Sales of daily consumer goods_Lag1", "Revenue_Lag1", "Revenue_Lag4", "CPI_YoY_Lag1", "Exchange_Rate_Lag1"]
target_rev = "Revenue"

In [7]:
X_rev = df_model[features_rev]
y_rev = df_model[target_rev]

In [8]:
model_rev = RandomForestRegressor(
        n_estimators=200, random_state=42
    )

model_rev.fit(X_rev, y_rev)

RandomForestRegressor(n_estimators=200, random_state=42)

In [9]:
features_op = ["Quarter", "Diapers Exp_Lag1", "Cosmetics_Lag1", "Cardboard_Lag1", "Sales of daily consumer goods_Lag1", "Operating Income_Lag1", "Operating Income_Lag4", "CPI_YoY_Lag1", "Exchange_Rate_Lag1", "Revenue", "COGS_Lag1"]
target_op = "Operating Income"

In [10]:
X_op = df_model[features_op]
y_op = df_model[target_op]

In [11]:
model_op = RandomForestRegressor(
        n_estimators=200, random_state=42
    )

model_op.fit(X_op, y_op)

RandomForestRegressor(n_estimators=200, random_state=42)

In [12]:
features_np = ["Quarter", "Diapers Exp_Lag1", "Cosmetics_Lag1", "Cardboard_Lag1", "Sales of daily consumer goods_Lag1", "Net Profit_Lag1", "Net Profit_Lag4", "CPI_YoY_Lag1", "Exchange_Rate_Lag1", "Revenue", "Operating Income", "COGS_Lag1"]
target_np = "Net Profit"

In [13]:
X_np = df_model[features_np]
y_np = df_model[target_np]

In [14]:
model_np = RandomForestRegressor(
        n_estimators=200, random_state=42
    )

model_np.fit(X_np, y_np)

RandomForestRegressor(n_estimators=200, random_state=42)

In [15]:
import pandas as pd
import numpy as np
import xgboost as xgb

# [Assuming 'df' is your current dataframe ending in June 2024]
# [Assuming models 'model_rev', 'model_op', 'model_np' are already trained on available data]

# 1. Define Simulation Parameters
# Last Actual Data Point = Q2 2024 (approx April 1st or June 30th depending on your file)
# We find it dynamically:
last_known_date = df['Date'].max()
target_date = pd.Timestamp("2026-12-30")    # The goal: Q4 2026

# Create History Buffer
history = df.copy()

print(f"--- STARTING MULTI-STEP FORECAST ---")
print(f"From: {last_known_date.date()} (Last Actual)")
print(f"To:   {target_date.date()} (Target)\n")
print(f"{'Date':<12} | {'Revenue':>12} | {'Op Income':>12} | {'Net Profit':>12}")
print("-" * 60)

# 2. The Loop
# We advance 3 months from the last known date to start the first prediction
current_date = last_known_date + pd.DateOffset(months=3)

while current_date <= target_date:
    # A. Get Lags from the growing 'history' dataframe
    row_lag1 = history.iloc[-1] # Previous Quarter (could be a prediction)
    row_lag4 = history.iloc[-4] # Previous Year (could be actual or prediction)

    # B. Construct Inputs
    input_row = {
        'Quarter': (current_date.month // 3) % 4 + 1,

        # --- MACRO LAGS (Carrying forward last known actuals) ---
        'Diapers Exp_Lag1': row_lag1['Diapers Exp'],
        'Cosmetics_Lag1': row_lag1['Cosmetics'],
        'CPI_YoY_Lag1': row_lag1['CPI_YoY'],
        'Exchange_Rate_Lag1': row_lag1['Exchange_Rate'],
        'Cardboard_Lag1': row_lag1['Cardboard'],
        'Sales of daily consumer goods_Lag1': row_lag1['Sales of daily consumer goods'],
        'COGS_Lag1': row_lag1['COGS'],


        # --- PERFORMANCE LAGS ---
        'Revenue_Lag1': row_lag1['Revenue'],
        'Revenue_Lag4': row_lag4['Revenue'],
        'Operating Income_Lag1': row_lag1['Operating Income'],
        'Operating Income_Lag4': row_lag4['Operating Income'],
        'Net Profit_Lag1': row_lag1['Net Profit'],
        'Net Profit_Lag4': row_lag4['Net Profit']
    }

    # C. Predict REVENUE
    # (Ensure the column order matches your training data exactly)
    input_df_rev = pd.DataFrame([input_row])[features_rev]
    pred_revenue = model_rev.predict(input_df_rev)[0]

    # D. Predict OP INCOME (Inject Revenue)
    input_row['Revenue'] = pred_revenue
    input_df_op = pd.DataFrame([input_row])[features_op]
    pred_op = model_op.predict(input_df_op)[0]

    # E. Predict NET PROFIT (Inject Op Income)
    input_row['Operating Income'] = pred_op
    input_df_np = pd.DataFrame([input_row])[features_np]
    pred_np = model_np.predict(input_df_np)[0]

    # F. Print & Check Targets
    if current_date == pd.Timestamp('2025-10-01') or current_date == pd.Timestamp('2026-10-01'):
         print(f"*** TARGET Q4 {current_date.year} ***")

    print(f"{current_date.date()} | {pred_revenue:,.0f} | {pred_op:,.0f} | {pred_np:,.0f}")

    # G. Append to History
    new_row = {
        'Date': current_date,
        'Revenue': pred_revenue,
        'Operating Income': pred_op,
        'Net Profit': pred_np,

        # IMPORTANT: Carry forward the macro vars so they exist for the next loop's Lag1
        'Diapers Exp': row_lag1['Diapers Exp'],
        'Cosmetics': row_lag1['Cosmetics'],
        'CPI_YoY': row_lag1['CPI_YoY'],
        'Exchange_Rate': row_lag1['Exchange_Rate'],
        'Cardboard': row_lag1['Cardboard'],
        'Sales of daily consumer goods': row_lag1['Sales of daily consumer goods'],
        'COGS': row_lag1['COGS'],

    }

    history = pd.concat([history, pd.DataFrame([new_row])], ignore_index=True)

    current_date = current_date + pd.DateOffset(months=3)

print("-" * 60)
print("Forecast Complete.")

--- STARTING MULTI-STEP FORECAST ---
From: 2024-06-30 (Last Actual)
To:   2026-12-30 (Target)

Date         |      Revenue |    Op Income |   Net Profit
------------------------------------------------------------
2024-09-30 | 243,779 | 34,252 | 23,990
2024-12-30 | 243,707 | 34,632 | 23,358
2025-03-30 | 243,925 | 34,549 | 23,790
2025-06-30 | 243,779 | 34,555 | 23,593
2025-09-30 | 243,779 | 34,552 | 23,593
2025-12-30 | 243,707 | 34,632 | 23,358
2026-03-30 | 243,925 | 34,468 | 23,378
2026-06-30 | 243,779 | 34,552 | 23,593
2026-09-30 | 243,779 | 34,552 | 23,593
2026-12-30 | 243,707 | 34,632 | 23,358
------------------------------------------------------------
Forecast Complete.


In [20]:
# ==========================================
# CALCULATION: AGGREGATE FY FROM 'HISTORY'
# ==========================================

# 1. Extract Year from the Date column in your 'history' dataframe
# (Make sure 'history' is the dataframe you used in the loop)
history['Year'] = pd.to_datetime(history['Date']).dt.year

# 2. Filter for only the forecast years (2025 and 2026)
df_fy_forecast = history[history['Year'].isin([2025, 2026])].copy()

# 3. Group by Year and Sum the quarterly values
df_fy_totals = df_fy_forecast.groupby('Year')[['Revenue', 'Operating Income', 'Net Profit']].sum().reset_index()

# 4. Display the Final Table
print("\n--- FINAL FULL YEAR (FY) PROJECTIONS (Millions JPY) ---")
print(df_fy_totals.to_string(index=False, float_format="{:,.0f}".format))

# 5. Calculate & Print Growth Rates
if len(df_fy_totals) == 2:
    p_2025 = df_fy_totals[df_fy_totals["Year"] == 2025].iloc[0]
    p_2026 = df_fy_totals[df_fy_totals["Year"] == 2026].iloc[0]

    print("\n--- YoY Growth (2026 vs 2025) ---")
    for col in ["Revenue", "Operating Income", "Net Profit"]:
        val_25 = p_2025[col]
        val_26 = p_2026[col]
        growth = ((val_26 - val_25) / val_25) * 100
        print(f"{col:<20} : {growth:>6.2f}%")


--- FINAL FULL YEAR (FY) PROJECTIONS (Millions JPY) ---
 Year  Revenue  Operating Income  Net Profit
 2025  975,189           138,288      94,333
 2026  975,189           138,203      93,921

--- YoY Growth (2026 vs 2025) ---
Revenue              :   0.00%
Operating Income     :  -0.06%
Net Profit           :  -0.44%
